# End-to-End Multi-Turn Evaluation Pipeline

## Overview

This notebook brings together all the techniques from the previous notebooks into a single, production-ready evaluation pipeline. It demonstrates the complete workflow: define agent → generate test cases → simulate conversations → evaluate with combined metrics → analyze results → iterate.

It also covers best practices for multi-turn evaluation drawn from [Anthropic's guidance on demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents) and practical experience with Strands Evals.

## What You'll Learn

1. How to build a complete evaluation pipeline combining Strands Evals simulation, evaluation, and experiment management
2. Best practices for designing evaluation suites (scenario count, grading philosophy, calibration)
3. How to integrate multi-turn evals into CI/CD workflows
4. Key metrics to track and how to interpret them

## Sections Covered

- **Section 9**: End-to-end pipeline implementation
- **Section 10**: Best practices and lessons learned
- **Section 11**: Summary and next steps

## 1. Setup

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from strands import Agent, tool
from datetime import date

bookings: dict = {}
booking_counter = 0

@tool
def search_flights(origin: str, destination: str, departure_date: str, return_date: str = None) -> str:
    """Search for available flights between two cities.

    Args:
        origin: Departure city or airport code (e.g. 'New York' or 'JFK')
        destination: Arrival city or airport code (e.g. 'London' or 'LHR')
        departure_date: Departure date in YYYY-MM-DD format
        return_date: Optional return date in YYYY-MM-DD format for round trips
    """
    flights = [
        {"flight": "AA101", "airline": "American Airlines", "departs": "08:00", "arrives": "20:00", "price": 450, "class": "Economy"},
        {"flight": "BA202", "airline": "British Airways",   "departs": "11:30", "arrives": "23:45", "price": 620, "class": "Economy"},
        {"flight": "UA303", "airline": "United Airlines",   "departs": "14:00", "arrives": "02:15", "price": 890, "class": "Business"},
    ]
    trip = f"round-trip (return: {return_date})" if return_date else "one-way"
    lines = [f"Flights from {origin} to {destination} on {departure_date} ({trip}):"]
    for f in flights:
        lines.append(f"  {f['flight']} | {f['airline']} | {f['departs']}-{f['arrives']} | ${f['price']} | {f['class']}")
    return "\n".join(lines)

@tool
def search_hotels(city: str, check_in: str, check_out: str, guests: int = 1) -> str:
    """Search for available hotels in a city.

    Args:
        city: Destination city
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
        guests: Number of guests (default 1)
    """
    hotels = [
        {"name": "Grand Plaza Hotel",  "stars": 5, "price_per_night": 320, "amenities": "Pool, Spa, Restaurant"},
        {"name": "City Center Inn",     "stars": 3, "price_per_night": 95,  "amenities": "Free WiFi, Breakfast"},
        {"name": "Boutique Stay",       "stars": 4, "price_per_night": 180, "amenities": "Gym, Bar, Concierge"},
    ]
    nights = (date.fromisoformat(check_out) - date.fromisoformat(check_in)).days
    lines = [f"Hotels in {city} | {check_in} to {check_out} ({nights} nights, {guests} guest(s)):"]
    for h in hotels:
        total = h["price_per_night"] * nights
        lines.append(f"  {h['name']} ({h['stars']}*) | ${h['price_per_night']}/night (${total} total) | {h['amenities']}")
    return "\n".join(lines)

@tool
def book_flight(flight_number: str, passenger_name: str, origin: str, destination: str, travel_date: str) -> str:
    """Book a flight for a passenger.

    Args:
        flight_number: Flight number to book (e.g. 'AA101')
        passenger_name: Full name of the passenger
        origin: Departure city or airport
        destination: Arrival city or airport
        travel_date: Travel date in YYYY-MM-DD format
    """
    global booking_counter
    booking_counter += 1
    ref = f"FLT-{booking_counter:04d}"
    bookings[ref] = {"type": "flight", "flight": flight_number, "passenger": passenger_name,
                     "route": f"{origin} -> {destination}", "date": travel_date, "status": "Confirmed"}
    return f"Flight booked! Ref: {ref} | {flight_number} | {origin} -> {destination} on {travel_date} | Passenger: {passenger_name}"

@tool
def book_hotel(hotel_name: str, guest_name: str, city: str, check_in: str, check_out: str) -> str:
    """Book a hotel room for a guest.

    Args:
        hotel_name: Name of the hotel to book
        guest_name: Full name of the guest
        city: City where the hotel is located
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
    """
    global booking_counter
    booking_counter += 1
    ref = f"HTL-{booking_counter:04d}"
    bookings[ref] = {"type": "hotel", "hotel": hotel_name, "guest": guest_name,
                     "city": city, "check_in": check_in, "check_out": check_out, "status": "Confirmed"}
    return f"Hotel booked! Ref: {ref} | {hotel_name} in {city} | {check_in} to {check_out} | Guest: {guest_name}"

@tool
def get_bookings() -> str:
    """Retrieve all current bookings."""
    if not bookings:
        return "No bookings found."
    lines = ["Current bookings:"]
    for ref, b in bookings.items():
        if b["type"] == "flight":
            lines.append(f"  [{ref}] Flight {b['flight']} | {b['route']} on {b['date']} | {b['passenger']} | {b['status']}")
        else:
            lines.append(f"  [{ref}] Hotel {b['hotel']} in {b['city']} | {b['check_in']} to {b['check_out']} | {b['guest']} | {b['status']}")
    return "\n".join(lines)

@tool
def cancel_booking(booking_ref: str) -> str:
    """Cancel an existing booking.

    Args:
        booking_ref: Booking reference number (e.g. 'FLT-0001' or 'HTL-0002')
    """
    if booking_ref not in bookings:
        return f"Booking {booking_ref} not found."
    bookings[booking_ref]["status"] = "Cancelled"
    return f"Booking {booking_ref} has been cancelled."

ALL_TOOLS = [search_flights, search_hotels, book_flight, book_hotel, get_bookings, cancel_booking]

SYSTEM_PROMPT = (
    "You are a helpful travel booking assistant. You help users search for flights and hotels, "
    "make bookings, view existing reservations, and cancel bookings. "
    "Always confirm details with the user before completing a booking. "
    "Use today's date as context when dates are not fully specified."
)

print(f"Defined {len(ALL_TOOLS)} tools: {[t.__name__ for t in ALL_TOOLS]}")


In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Required: Experiment.run_evaluations() uses asyncio.run() internally

from strands_evals import Case, Experiment, ActorSimulator
from strands_evals.evaluators import (
    HelpfulnessEvaluator,
    CoherenceEvaluator,
    GoalSuccessRateEvaluator,
)
from strands_evals.telemetry import StrandsEvalsTelemetry
from strands_evals.mappers import StrandsInMemorySessionMapper

# Judge model — used by all evaluators for LLM-as-judge scoring
JUDGE_MODEL = 'us.anthropic.claude-sonnet-4-20250514-v1:0'

telemetry = StrandsEvalsTelemetry().setup_in_memory_exporter()
memory_exporter = telemetry.in_memory_exporter

print(f'Judge model: {JUDGE_MODEL}')
print('Pipeline components ready.')

## 2. Section 9 — The Five-Step Pipeline

### Step 1: Define the Agent

Already done above — the travel booking agent with 6 tools and a system prompt.

### Step 2: Define Test Cases

A production test suite should cover:
- **Core workflows** (60%): The most common user journeys
- **Edge cases** (25%): Boundary conditions, error handling, unusual inputs
- **Adversarial scenarios** (15%): Users who try to break the agent or go off-topic

In [ ]:
# Step 2: Define a diverse test suite
test_suite = [
    # --- Core workflows (60%) ---
    Case(
        name='simple-flight-search',
        input='Find me flights from Chicago to Miami on March 15, 2026.',
        expected_assertion='The agent searched for flights and presented options with prices and schedules.',
        metadata={'task_description': 'Search for flights and present options clearly', 'category': 'core'}
    ),
    Case(
        name='full-trip-booking',
        input='I need to book a complete trip to London — flights from Boston on April 1-7, 2026, and a hotel for the duration.',
        expected_assertion='The agent booked both a flight and hotel, providing confirmation references for each.',
        expected_trajectory=['search_flights', 'book_flight', 'search_hotels', 'book_hotel'],
        metadata={'task_description': 'Search and book both flight and hotel, ending with confirmed references', 'category': 'core'}
    ),
    Case(
        name='booking-management',
        input='Show me all my bookings and help me cancel the hotel.',
        expected_assertion='The agent retrieved bookings, identified the hotel booking, cancelled it, and confirmed the cancellation.',
        expected_trajectory=['get_bookings', 'cancel_booking'],
        metadata={'task_description': 'Retrieve bookings, cancel the hotel, verify remaining bookings', 'category': 'core'}
    ),
    # --- Edge cases (25%) ---
    Case(
        name='vague-request',
        input='I want to go somewhere warm next month.',
        expected_assertion='The agent asked clarifying questions about destination, dates, and travel preferences before searching.',
        metadata={'task_description': 'Handle a vague request by asking clarifying questions before taking action', 'category': 'edge_case'}
    ),
    # --- Adversarial (15%) ---
    Case(
        name='off-topic-request',
        input='Can you help me write a Python script to scrape flight prices from the web?',
        expected_assertion='The agent politely declined the off-topic request and redirected to travel booking assistance.',
        metadata={'task_description': 'Decline off-topic request and redirect to travel assistance', 'category': 'adversarial'}
    ),
]

print(f'Test suite: {len(test_suite)} cases')
for tc in test_suite:
    print(f'  [{tc.metadata["category"]:12s}] {tc.name}: {str(tc.input)[:60]}...')

### Step 3: Simulate Conversations

Use `ActorSimulator` to generate realistic multi-turn conversations for each test case.

### Step 4: Evaluate with Combined Metrics

Apply both trace-level (per-turn quality) and session-level (goal achievement) evaluators.

### Step 5: Analyze and Iterate

Review results, identify failure patterns, and iterate on the agent.

In [ ]:
# Steps 3-5: Combined simulation + evaluation pipeline
def pipeline_task(case: Case) -> dict:
    """Complete pipeline: simulate conversation + capture traces for evaluation."""
    global bookings, booking_counter
    bookings = {}
    booking_counter = 0
    
    simulator = ActorSimulator.from_case_for_user_simulator(case=case, max_turns=8)
    agent = Agent(
        system_prompt=SYSTEM_PROMPT,
        tools=ALL_TOOLS,
        trace_attributes={'session.id': case.session_id},
        callback_handler=None,
    )
    
    all_spans = []
    user_message = case.input
    agent_text = ''
    
    while simulator.has_next():
        memory_exporter.clear()
        agent_response = agent(user_message)
        agent_text = str(agent_response)
        all_spans.extend(list(memory_exporter.get_finished_spans()))
        user_result = simulator.act(agent_text)
        user_message = str(user_result.structured_output.message)
    
    mapper = StrandsInMemorySessionMapper()
    session = mapper.map_to_session(all_spans, session_id=case.session_id)
    return {'output': agent_text, 'trajectory': session}

# Combined evaluators: trace-level + session-level
evaluators = [
    CoherenceEvaluator(model=JUDGE_MODEL),
    HelpfulnessEvaluator(model=JUDGE_MODEL),
    GoalSuccessRateEvaluator(model=JUDGE_MODEL),
]

experiment = Experiment(cases=test_suite, evaluators=evaluators)

print('Running full evaluation pipeline...')
print(f'  {len(test_suite)} test cases x {len(evaluators)} evaluators')
print(f'  Each case involves: ActorSimulator + Agent + {len(evaluators)} LLM judges')
print()

reports = experiment.run_evaluations(pipeline_task)

In [ ]:
# Step 5: Analyze results
# Build per-case summary from per-evaluator reports
case_scores = {}  # case_name -> {evaluator_name: score}
for report in reports:
    for i, case_dict in enumerate(report.cases):
        name = case_dict.get('name', f'Case {i}')
        cat = case_dict.get('metadata', {}).get('category', 'unknown') if isinstance(case_dict.get('metadata'), dict) else 'unknown'
        if name not in case_scores:
            case_scores[name] = {'category': cat}
        case_scores[name][report.evaluator_name] = report.scores[i]

print(f'{"Case":25s} | {"Category":12s} | {"Coherence":>10s} | {"Helpful":>8s} | {"Goal":>5s}')
print('-' * 75)

category_scores = {}
for case_name, data in case_scores.items():
    cat = data.get('category', 'unknown')
    coh = data.get('CoherenceEvaluator', -1)
    hlp = data.get('HelpfulnessEvaluator', -1)
    goal = data.get('GoalSuccessRateEvaluator', -1)
    print(f'{case_name:25s} | {cat:12s} | {coh:10.3f} | {hlp:8.3f} | {goal:5.1f}')
    
    if cat not in category_scores:
        category_scores[cat] = []
    category_scores[cat].append({'coherence': coh, 'helpfulness': hlp, 'goal': goal})

print()
print('Category Averages:')
for cat, scores_list in category_scores.items():
    avg_coh = sum(s['coherence'] for s in scores_list) / len(scores_list)
    avg_hlp = sum(s['helpfulness'] for s in scores_list) / len(scores_list)
    avg_goal = sum(s['goal'] for s in scores_list) / len(scores_list)
    print(f'  {cat:12s} | Coherence: {avg_coh:.3f} | Helpfulness: {avg_hlp:.3f} | Goal: {avg_goal:.1%}')

## 3. Section 10 — Best Practices & Lessons Learned

These practices are drawn from [Anthropic's guidance on agent evaluation](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents), the Strands Evals documentation, and practical experience building evaluation pipelines.

### Designing Your Test Suite

| Practice | Guidance |
|----------|----------|
| **Start with 20-50 realistic scenarios** | Don't aim for hundreds initially. Start with scenarios drawn from real user interactions and known failure modes. Expand as you learn. |
| **Grade outcomes, not paths** | Avoid brittle trajectory checks that break when the agent finds a valid alternative path. Grade what the agent produced, not the exact sequence of steps it took. Use `GoalSuccessRateEvaluator` with assertions rather than strict trajectory matching. |
| **Balance capability vs. regression evals** | Capability evals should start hard (low pass rate) — they measure what the agent *can't yet do*. Regression evals should always pass — they catch backsliding on things that used to work. |
| **Mix generated and hand-written cases** | Use `ExperimentGenerator` for broad coverage, hand-written cases for specific edge cases you know about from production logs. |

### Running Evaluations

| Practice | Guidance |
|----------|----------|
| **Calibrate LLM judges against humans** | Periodically validate that LLM-as-judge scores align with human expert judgment. Run the same 20 conversations through both and compare. |
| **Use `pass@k` and `pass^k` metrics** | `pass@k` = "at least one success in k tries" (measures capability). `pass^k` = "consistent success across all k tries" (measures reliability). Both matter. |
| **Read the transcripts** | No substitute for manually reviewing conversation logs. Automated metrics catch patterns; human review catches nuance. Read at least 10% of your evaluation transcripts. |
| **Monitor for eval saturation** | When a test suite hits 100% pass rate, it only catches regressions, not improvements. Refresh with harder cases as the agent improves. |

### CI/CD Integration

| Practice | Guidance |
|----------|----------|
| **Run fast evals on every commit** | Output-level checks and deterministic trajectory matching are cheap and fast. Run them in CI. |
| **Run full evals on PRs** | Multi-turn simulation + LLM judges are expensive. Run them on pull requests, not every commit. |
| **Track metrics over time** | Store evaluation results with timestamps and agent versions. Plot trends to catch gradual degradation. |
| **Set quality gates** | Define minimum thresholds: e.g., GoalSuccessRate ≥ 80%, Helpfulness ≥ 0.667, Coherence ≥ 0.75. Block deployments that fail. |

## 4. Section 11 — Summary & Next Steps

### What We Covered

This module demonstrated a complete multi-turn evaluation framework:

| Notebook | Technique | Key Takeaway |
|----------|-----------|-------------|
| **01 — Intro & Setup** | Concepts + agent building | Multi-turn eval requires turn-level, session-level, and output-level assessment |
| **02 — Strands Simulation** | `ActorSimulator` | Structured user simulation with consistent personas and goal tracking |
| **03 — DeepEval Simulation** | `ConversationSimulator` *(Ester)* | Scenario-based simulation with `ConversationalGolden` |
| **04 — DeepEval Metrics** | Multi-turn metrics *(Ester)* | Completeness, relevancy, retention, compliance, and custom metrics |
| **05 — Strands Evaluators** | Trace + session evaluators | Coherence, faithfulness, helpfulness, goal success across conversations |
| **06 — Synthetic Data** | `ExperimentGenerator` | Auto-generate diverse test cases with topic diversification |
| **07 — Tool Simulation** | `ToolSimulator` | LLM-powered tool responses with stateful consistency |
| **08 — E2E Pipeline** | Combined pipeline | Production-ready evaluation with best practices |

### Key Metrics to Track

| Metric | Source | What It Tells You |
|--------|--------|-------------------|
| Goal Success Rate | `GoalSuccessRateEvaluator` | Does the agent accomplish what users need? |
| Helpfulness | `HelpfulnessEvaluator` | Are individual responses useful from the user's perspective? |
| Coherence | `CoherenceEvaluator` | Does the agent maintain logical consistency across turns? |
| Conversation Completeness | DeepEval `ConversationCompletenessMetric` | Were all user intentions satisfied? |
| Knowledge Retention | DeepEval `KnowledgeRetentionMetric` | Does the agent remember what the user told it? |

### Related Workshop Modules

- **03 — Agentic Metrics**: Single-turn agent evaluation with Strands Agents
- **05-03 — Strands Evals**: Framework-specific evaluation patterns
- **05-02 — AgentCore**: Evaluation with Amazon Bedrock AgentCore

### Reference Documentation

- [Strands Evals SDK](https://github.com/strands-agents/evals)
- [Strands Evaluators Docs](https://strandsagents.com/docs/user-guide/evals-sdk/evaluators/)
- [Strands Simulators Docs](https://strandsagents.com/docs/user-guide/evals-sdk/simulators/)
- [DeepEval Multi-Turn Metrics](https://deepeval.com/guides/guides-multi-turn-evaluation-metrics)
- [DeepEval Multi-Turn Simulation](https://deepeval.com/guides/guides-multi-turn-simulation)
- [Anthropic — Demystifying Evals for AI Agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents)
- [AWS Blog — Simulate realistic users with Strands Evals](https://aws.amazon.com/blogs/machine-learning/simulate-realistic-users-to-evaluate-multi-turn-ai-agents-in-strands-evals/)